# Algoritmos - Actividad Guiada 1

**Nombre:** Medina Toapanta Graciela Elizabeth <br>

**URL:**  

**http:**  https://github.com/gemedinat/MIAR_ALGORITMOS_DE_OPTIMIZACION.git 


## Torres de Hanoi con Divide y vencerás

In [6]:
def Torres_Hanoi(N, desde, hasta):
  if N ==1 :
    print("Lleva la ficha " ,desde , " hasta ", hasta )

  else:
    #Torres_Hanoi(N-1, desde, 6-desde-hasta )
    Torres_Hanoi(N-1, desde, 6-desde-hasta )
    print("Lleva la ficha " ,desde , " hasta ", hasta )
    #Torres_Hanoi(N-1,6-desde-hasta, hasta )
    Torres_Hanoi(N-1, 6-desde-hasta  , hasta )


Torres_Hanoi(3, 1 , 3)

Lleva la ficha  1  hasta  3
Lleva la ficha  1  hasta  2
Lleva la ficha  3  hasta  2
Lleva la ficha  1  hasta  3
Lleva la ficha  2  hasta  1
Lleva la ficha  2  hasta  3
Lleva la ficha  1  hasta  3


**Propuesta de ajuste:**

1. Se  calcula `aux = 6 - desde - hasta` una sola vez y con nombre, lo que hace evidente la estructura *divide y vencerás* (mover N-1 al auxiliar, mover la mayor, mover N-1 al destino).
2. Separar el algoritmo de la entrada/salida para que sea testeable y reutilizable.
3. El  número de movimientos debe ser exactamente $2^N - 1$ (cota mínima demostrable), lo que nos permite validar la implementación. La complejidad temporal es $O(2^N)$ y es inevitable: hay que generar cada movimiento.

In [10]:
def Torres_Ajustadas(N, desde, hasta, movimientos=None):
    if movimientos is None:
        movimientos = []
    if N == 1:
        movimientos.append((desde, hasta))
    else:
        aux = 6 - desde - hasta          # la tercera varilla (1+2+3 = 6)
        Torres_Ajustadas(N-1, desde, aux, movimientos)   # subproblema 1
        movimientos.append((desde, hasta))           # caso base sobre el disco mayor
        Torres_Ajustadas(N-1, aux, hasta, movimientos)   # subproblema 2
    return movimientos

movs = Torres_Ajustadas(3, 1, 3)
for desde, hasta in movs:
    print(f"Lleva la ficha {desde} hasta {hasta}")


Lleva la ficha 1 hasta 3
Lleva la ficha 1 hasta 2
Lleva la ficha 3 hasta 2
Lleva la ficha 1 hasta 3
Lleva la ficha 2 hasta 1
Lleva la ficha 2 hasta 3
Lleva la ficha 1 hasta 3


In [13]:
# Verificación del mínimo teórico de movimientos es 2^N - 1
for n in range(1, 11):
    assert len(Torres_Ajustadas(n, 1, 3)) == 2**n - 1
print("Para todo N probado, movimientos = 2^N - 1")

Para todo N probado, movimientos = 2^N - 1


In [2]:
#Sucesión_de_Fibonacci
#https://es.wikipedia.org/wiki/Sucesi%C3%B3n_de_Fibonacci
#Calculo del termino n-simo de la suscesión de Fibonacci
def Fibonacci(N:int):
  if N < 2:
    return 1
  else:
    return Fibonacci(N-1)+Fibonacci(N-2)

Fibonacci(5)

8

**Propuesta Ajuste:**

La recursión directa recalcula los mismos subproblemas una y otra vez: `Fibonacci(N-2)` se calcula dentro de `Fibonacci(N-1)` y otra vez de forma independiente. La complejidad es $O(\varphi^N)$ con $\varphi \approx 1.618$ — calcular `Fibonacci(40)` ya tarda segundos.
Guardamos cada resultado la primera vez que se calcula. 

In [14]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fibonacci_memo(N: int):
    if N < 2:
        return 1
    return fibonacci_memo(N-1) + fibonacci_memo(N-2)

fibonacci_memo(100) 

573147844013817084101

## Devolución de cambio por técnica voraz

In [3]:
def cambio_monedas(N, SM):
  SOLUCION = [0]*len(SM)   #SOLUCION = [0,0,0,0,..]
  ValorAcumulado = 0

  for i,valor in enumerate(SM):
    monedas =  (N-ValorAcumulado)//valor
    SOLUCION[i] = monedas
    ValorAcumulado = ValorAcumulado + monedas*valor

    if ValorAcumulado == N:
      return SOLUCION


cambio_monedas(15,[25,10,5,1])

[0, 1, 1, 0]

**Propuesta de mejora:**

Se corrigen algunos problemas de código detectados:
1. Si  el sistema no permite completar el cambio (p. ej. `cambio_monedas(11, [25,10,5])`), la función devuelve `None` sin avisar.
2. Si `SM` no viene ordenado de mayor a menor, el resultado es incorrecto.
3. Solo lo es para sistemas *canónicos* (como el del euro). Con `SM = [25, 10, 1]` y `N = 30`, el voraz usa 6 monedas (25+1+1+1+1+1) cuando el óptimo son 3 (10+10+10).


In [15]:
def cambio_monedas_voraz(N, SM):
    SM = sorted(SM, reverse=True)               # robustez frente al orden de entrada
    solucion = [0]*len(SM)
    restante = N
    for i, valor in enumerate(SM):
        solucion[i], restante = divmod(restante, valor)
    if restante != 0:
        return None                             # fallo explícito: no hay solución voraz
    return dict(zip(SM, solucion))


def cambio_monedas_dp(N, SM):
    """Mínimo número de monedas garantizado (programación dinámica)."""
    INF = float('inf')
    mejor = [0] + [INF]*N          # mejor[c] = mínimo de monedas para el importe c
    moneda_usada = [0]*(N+1)
    for c in range(1, N+1):
        for valor in SM:
            if valor <= c and mejor[c-valor] + 1 < mejor[c]:
                mejor[c] = mejor[c-valor] + 1
                moneda_usada[c] = valor
    if mejor[N] == INF:
        return None
    solucion, c = {}, N
    while c > 0:
        v = moneda_usada[c]
        solucion[v] = solucion.get(v, 0) + 1
        c -= v
    return solucion


print("Sistema canónico, N=15:", cambio_monedas_voraz(15, [25, 10, 5, 1]))
print("Sin solución, N=11, SM=[25,10,5]:", cambio_monedas_voraz(11, [25, 10, 5]))

g = cambio_monedas_voraz(30, [25, 10, 1])
d = cambio_monedas_dp(30, [25, 10, 1])
print(f"\nN=30, SM=[25,10,1] -> Voraz: {g} ({sum(g.values())} monedas) | DP: {d} ({sum(d.values())} monedas)")

Sistema canónico, N=15: {25: 0, 10: 1, 5: 1, 1: 0}
Sin solución, N=11, SM=[25,10,5]: None

N=30, SM=[25,10,1] -> Voraz: {25: 1, 10: 0, 1: 5} (6 monedas) | DP: {10: 3} (3 monedas)


## N-Reinas por técnica de vueta atrás


In [4]:
def escribe(S):
  n = len(S)
  for x in range(n):
    print("")
    for i in range(n):
      if S[i] == x+1:
        print(" X " , end="")
      else:
        print(" - ", end="")


def es_prometedora(SOLUCION,etapa):
  #print(SOLUCION)
  #Si la solución tiene dos valores iguales no es valida => Dos reinas en la misma fila
  for i in range(etapa+1):
    #print("El valor " + str(SOLUCION[i]) + " está " +  str(SOLUCION.count(SOLUCION[i])) + " veces")
    if SOLUCION.count(SOLUCION[i]) > 1:
      return False

    #Verifica las diagonales
    for j in range(i+1, etapa +1 ):
      #print("Comprobando diagonal de " + str(i) + " y " + str(j))
      if abs(i-j) == abs(SOLUCION[i]-SOLUCION[j]) : return False
  return True



def reinas(N, solucion=[], etapa=0):
  if len(solucion) == 0:
      solucion=[0 for i in range(N)]

  for i in range(1, N+1):
    solucion[etapa] = i

    if es_prometedora(solucion, etapa):
      if etapa == N-1:
        print(solucion)
        #escribe(solucion)
        print()
      else:
        reinas(N, solucion, etapa+1)
    else:
      None

    solucion[etapa] = 0

reinas(8)

[1, 5, 8, 6, 3, 7, 2, 4]

[1, 6, 8, 3, 7, 4, 2, 5]

[1, 7, 4, 6, 8, 2, 5, 3]

[1, 7, 5, 8, 2, 4, 6, 3]

[2, 4, 6, 8, 3, 1, 7, 5]

[2, 5, 7, 1, 3, 8, 6, 4]

[2, 5, 7, 4, 1, 8, 6, 3]

[2, 6, 1, 7, 4, 8, 3, 5]

[2, 6, 8, 3, 1, 4, 7, 5]

[2, 7, 3, 6, 8, 5, 1, 4]

[2, 7, 5, 8, 1, 4, 6, 3]

[2, 8, 6, 1, 3, 5, 7, 4]

[3, 1, 7, 5, 8, 2, 4, 6]

[3, 5, 2, 8, 1, 7, 4, 6]

[3, 5, 2, 8, 6, 4, 7, 1]

[3, 5, 7, 1, 4, 2, 8, 6]

[3, 5, 8, 4, 1, 7, 2, 6]

[3, 6, 2, 5, 8, 1, 7, 4]

[3, 6, 2, 7, 1, 4, 8, 5]

[3, 6, 2, 7, 5, 1, 8, 4]

[3, 6, 4, 1, 8, 5, 7, 2]

[3, 6, 4, 2, 8, 5, 7, 1]

[3, 6, 8, 1, 4, 7, 5, 2]

[3, 6, 8, 1, 5, 7, 2, 4]

[3, 6, 8, 2, 4, 1, 7, 5]

[3, 7, 2, 8, 5, 1, 4, 6]

[3, 7, 2, 8, 6, 4, 1, 5]

[3, 8, 4, 7, 1, 6, 2, 5]

[4, 1, 5, 8, 2, 7, 3, 6]

[4, 1, 5, 8, 6, 3, 7, 2]

[4, 2, 5, 8, 6, 1, 3, 7]

[4, 2, 7, 3, 6, 8, 1, 5]

[4, 2, 7, 3, 6, 8, 5, 1]

[4, 2, 7, 5, 1, 8, 6, 3]

[4, 2, 8, 5, 7, 1, 3, 6]

[4, 2, 8, 6, 1, 3, 5, 7]

[4, 6, 1, 5, 2, 8, 3, 7]

[4, 6, 8, 2, 7, 1, 3, 5]

[4, 6, 8, 3,

**Propuesta de mejora:**

Comprobar si una posición es prometedora pasa a ser $O(1)$. Dos reinas en (fila, col) comparten diagonal . La poda es idéntica (es el mismo árbol de búsqueda)pero cada nodo se evalúa en tiempo constante.



In [16]:
def reinas_optimizado(N):
    soluciones = []
    columnas, diag1, diag2 = set(), set(), set()
    solucion = [0]*N

    def backtrack(fila):
        if fila == N:
            soluciones.append(solucion.copy())
            return
        for col in range(1, N+1):
            if col in columnas or (fila+col) in diag1 or (fila-col) in diag2:
                continue                                  # poda en O(1)
            solucion[fila] = col
            columnas.add(col); diag1.add(fila+col); diag2.add(fila-col)
            backtrack(fila + 1)
            columnas.discard(col); diag1.discard(fila+col); diag2.discard(fila-col)

    backtrack(0)
    return soluciones


soluciones = reinas_optimizado(8)
print(f"N=8: {len(soluciones)} soluciones (valor conocido: 92)")
print("Primera solución:", soluciones[0])

# Comparación de tiempos con la versión original
import time

def es_prometedora(SOLUCION, etapa):
  for i in range(etapa+1):
    if SOLUCION.count(SOLUCION[i]) > 1: return False
    for j in range(i+1, etapa+1):
      if abs(i-j) == abs(SOLUCION[i]-SOLUCION[j]): return False
  return True

def reinas_original(N, solucion=None, etapa=0, acumulador=None):
  if solucion is None:
      solucion = [0]*N
      acumulador = []
  for i in range(1, N+1):
    solucion[etapa] = i
    if es_prometedora(solucion, etapa):
      if etapa == N-1:
        acumulador.append(solucion.copy())
      else:
        reinas_original(N, solucion, etapa+1, acumulador)
    solucion[etapa] = 0
  return acumulador

for n in (8, 10):
    t0 = time.time(); reinas_original(n);   t1 = time.time()
    t2 = time.time(); reinas_optimizado(n); t3 = time.time()
    print(f"N={n}: original {t1-t0:.3f}s | optimizado {t3-t2:.3f}s")

N=8: 92 soluciones (valor conocido: 92)
Primera solución: [1, 5, 8, 6, 3, 7, 2, 4]
N=8: original 0.072s | optimizado 0.004s
N=10: original 2.150s | optimizado 0.093s


## Viaje por el rio. Programación dinámica

In [5]:
TARIFAS = [
[0,5,4,3,999,999,999],
[999,0,999,2,3,999,11],
[999,999, 0,1,999,4,10],
[999,999,999, 0,5,6,9],
[999,999, 999,999,0,999,4],
[999,999, 999,999,999,0,3],
[999,999,999,999,999,999,0]
]



################################################################
def Precios(TARIFAS):
################################################################
  #Total de Nodos
  N = len(TARIFAS[0])

  #Inicialización de la tabla de precios
  PRECIOS = [ [9999]*N for i in [9999]*N]
  RUTA = [ [""]*N for i in [""]*N]

  for i in range(0,N-1):
    RUTA[i][i] = i             #Para ir de i a i se "pasa por i"
    PRECIOS[i][i] = 0          #Para ir de i a i se se paga 0
    for j in range(i+1, N):
      MIN = TARIFAS[i][j]
      RUTA[i][j] = i

      for k in range(i, j):
        if PRECIOS[i][k] + TARIFAS[k][j] < MIN:
            MIN = min(MIN, PRECIOS[i][k] + TARIFAS[k][j] )
            RUTA[i][j] = k          #Anota que para ir de i a j hay que pasar por k
        PRECIOS[i][j] = MIN

  return PRECIOS,RUTA
################################################################


PRECIOS,RUTA = Precios(TARIFAS)
#print(PRECIOS[0][6])

print("PRECIOS")
for i in range(len(TARIFAS)):
  print(PRECIOS[i])

print("\nRUTA")
for i in range(len(TARIFAS)):
  print(RUTA[i])

#Determinar la ruta con Recursividad
def calcular_ruta(RUTA, desde, hasta):
  if desde == hasta:
    #print("Ir a :" + str(desde))
    return ""
  else:
    return str(calcular_ruta( RUTA, desde, RUTA[desde][hasta])) +  \
                ',' + \
                str(RUTA[desde][hasta] \
              )

print("\nLa ruta es:")
calcular_ruta(RUTA, 0,6)

PRECIOS
[0, 5, 4, 3, 8, 8, 11]
[9999, 0, 999, 2, 3, 8, 7]
[9999, 9999, 0, 1, 6, 4, 7]
[9999, 9999, 9999, 0, 5, 6, 9]
[9999, 9999, 9999, 9999, 0, 999, 4]
[9999, 9999, 9999, 9999, 9999, 0, 3]
[9999, 9999, 9999, 9999, 9999, 9999, 9999]

RUTA
[0, 0, 0, 0, 1, 2, 5]
['', 1, 1, 1, 1, 3, 4]
['', '', 2, 2, 3, 2, 5]
['', '', '', 3, 3, 3, 3]
['', '', '', '', 4, 4, 4]
['', '', '', '', '', 5, 5]
['', '', '', '', '', '', '']

La ruta es:


',0,2,5'

**Mejora Propuesta:**

Se corrigen los siguientes problemas:

1. El bucle exterior llega hasta N-2, así que PRECIOS[N-1][N-1] queda en 9999 en vez de 0.
2. Una ruta compuesta puede sumar más de 999 y parecer más cara que un trayecto inexistente. 
3. MIN = min(MIN, ...) dentro de un if que ya garantiza que el nuevo valor es menor, y PRECIOS[i][j] = MIN repetido dentro del bucle interno.
4. La ruta reconstruida es una cadena ',0,2,5' con una coma inicial y sin el nodo destino. 

In [22]:
NA = float('inf')

TARIFAS = [
[0,   5,   4,   3,   NA, NA, NA],
[NA, 0,   NA, 2,   3,   NA, 11 ],
[NA, NA, 0,   1,   NA, 4,   10 ],
[NA, NA, NA, 0,   5,   6,   9  ],
[NA, NA, NA, NA, 0,   NA, 4  ],
[NA, NA, NA, NA, NA, 0,   3  ],
[NA, NA, NA, NA, NA, NA, 0  ],
]

def precios(TARIFAS):
    N = len(TARIFAS)
    PRECIOS = [[INF]*N for _ in range(N)]
    RUTA = [[None]*N for _ in range(N)]

    for i in range(N):                 # Para toda la diagonal, incluido el último nodo
        PRECIOS[i][i] = 0
        RUTA[i][i] = i

    for i in range(N-1):
        for j in range(i+1, N):
            MIN = TARIFAS[i][j]        # opción directa i -> j
            RUTA[i][j] = i
            for k in range(i+1, j):    # Cuando pasa por un embarcadero intermedio k
                if PRECIOS[i][k] + TARIFAS[k][j] < MIN:
                    MIN = PRECIOS[i][k] + TARIFAS[k][j]
                    RUTA[i][j] = k
            PRECIOS[i][j] = MIN
    return PRECIOS, RUTA


# Reconstruimos la lista original
def calcular_ruta(RUTA, desde, hasta):
    if desde == hasta:
        return [desde]
    return calcular_ruta(RUTA, desde, RUTA[desde][hasta]) + [hasta]


PRECIOS, RUTA = precios(TARIFAS)

print("PRECIOS")
for fila in PRECIOS: print(fila)

ruta = calcular_ruta(RUTA, 0, 6)
print(f"\nCoste mínimo 0 -> 6: {PRECIOS[0][6]}")
print("Ruta:", " -> ".join(map(str, ruta)))

PRECIOS
[0, 5, 4, 3, 8, 8, 11]
[inf, 0, inf, 2, 3, 8, 7]
[inf, inf, 0, 1, 6, 4, 7]
[inf, inf, inf, 0, 5, 6, 9]
[inf, inf, inf, inf, 0, inf, 4]
[inf, inf, inf, inf, inf, 0, 3]
[inf, inf, inf, inf, inf, inf, 0]

Coste mínimo 0 -> 6: 11
Ruta: 0 -> 2 -> 5 -> 6


**Referencias Bibliográficas:**

>Aho, A. V., Hopcroft, J. E., & Ullman, J. D. (1983). Data structures and algorithms. Addison-Wesley.

>Bellman, R. (1957). Dynamic programming. Princeton University Press.

>Brassard, G., & Bratley, P. (1997). Fundamentos de algoritmia. Prentice Hall.

>Cormen, T. H., Leiserson, C. E., Rivest, R. L., & Stein, C. (2022). Introduction to algorithms (4th ed.). MIT Press.

>Kleinberg, J., & Tardos, É. (2006). Algorithm design. Pearson/Addison-Wesley.

>Levitin, A. (2012). Introduction to the design and analysis of algorithms (3rd ed.). Pearson.

>Python Software Foundation. (2026). The Python standard library: functools — Higher-order functions and operations on callable objects. https://docs.python.org/3/library/functools.html

>Skiena, S. S. (2020). The algorithm design manual (3rd ed.). Springer.